# Streaming

### Importing the necessary Modules & Packages

In [6]:
import boto3
import json

In [7]:
# Creating the bedrock client
anthropic_bedrock = boto3.client('bedrock-runtime', region_name = 'us-west-2')

# Model ID
model_id = 'global.anthropic.claude-sonnet-4-6'

# System Prompt
system_prompt = ''

# Temperature
temperature = 1.0

In [20]:
# Defining the function for user message

def user_message(text):
    user_message = {
        'role' : 'user',
        'content' : [
            { 'text' : text }
        ]
    }

    return user_message

# Defining the function for assistance message
def assistant_message(text):
    assistant_message = {
        'role' : 'assistant',
        'content' : [
            { 'text' : text }
        ]
    }

    return assistant_message

# Function to invoke the model and return the assistant response
def chat(user_message, temperature = 1.0):

    
    response = anthropic_bedrock.converse(
        modelId = model_id,
        messages = [user_message],
        inferenceConfig = {
            'maxTokens' : 512,
            'temperature' : temperature
        }
    )

    return response['output']['message']['content'][0]['text']

In [21]:
input_message = "Write a brief story not more than 3 sentence."

user_message = create_message(input_message)

model_response = chat(user_message)  # response from chat funtion defined earlier will be saved to answer variable - this is assistant message

print(model_response)


Here is a brief story:

Sara found an old, dusty letter hidden beneath the floorboards of her grandmother's attic. With trembling hands, she unfolded the yellowed paper and discovered a long-kept family secret that changed everything she thought she knew. She sat alone in the silence, tears streaming down her face, as the past and present collided in her heart.


### Streaming

In [25]:
# Creating the bedrock client
anthropic_bedrock = boto3.client('bedrock-runtime', region_name = 'us-west-2')

# Model ID
model_id = 'global.anthropic.claude-sonnet-4-6'

# System Prompt
system_prompt = ''

# Temperature
temperature = 1.0

In [27]:
# Defining the function for user message
messages = []

def add_user_message(messages, text):
    user_message = {
        'role' : 'user',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(user_message)
    return user_message

# Defining the function for assistance message
def add_assistant_message(messages, text):
    assistant_message = {
        'role' : 'assistant',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(assistant_message)

# Function to invoke the model and return the assistant response
def chat(user_message, temperature = 1.0):
    
    response = anthropic_bedrock.converse(
        modelId = model_id,
        messages = messages,
        inferenceConfig = {
            'maxTokens' : 512,
            'temperature' : temperature
        }
    )

    return response['output']['message']['content'][0]['text']

In [28]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "What's 1+1?"
add_user_message(messages, "What's 1+1?")

# Pass the list of messages into chat to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in the user's followup question
add_user_message(messages, "And 3 more added to that?")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
print(answer)

2 + 3 = **5**


In [29]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")
response = anthropic_bedrock.converse_stream(messages=messages, modelId=model_id)

for event in response["stream"]:
    print(event)

{'messageStart': {'role': 'assistant'}}
{'contentBlockDelta': {'delta': {'text': 'Here'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' is'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' a 1 sentence description of a'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' fake database:\n\n**'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': '"'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'N'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'ov'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'a'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': 'Base'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' is a fictional'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': ' cloud'}, 'contentBlockIndex': 0}}
{'contentBlockDelta': {'delta': {'text': '-based database management'}, 'contentBlockIndex'

In [32]:
add_user_message(messages, "Write a 1 sentence dsecription of a fake jokes database")

response  = anthropic_bedrock.converse_stream(messages = messages, modelId = model_id)

for event in response['stream']:
    if "contentBlockDelta" in event:
        chunk = event['contentBlockDelta']['delta']['text']
        print(chunk, end ='')

Here is a one sentence description of a fake jokes database:

**JokesterDB** is a fictional database containing over 10 million groan-worthy puns, dad jokes, and one-liners, categorized by humor level, topic, and audience age rating.

In [31]:
add_user_message(messages, "Write a 1 sentence description of a fake database")
response = anthropic_bedrock.converse_stream(messages=messages, modelId=model_id)

text = ""
for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"]["text"]
        print(chunk, end="")
        text += chunk

print("\n\nTotal Message:\n" + text)



Here is a one sentence description of a fake database:

**"The GlobalPet Registry is a fictional database containing records of over 50 million registered pets worldwide, including their names, species, medical histories, and owner information."**

Total Message:
Here is a one sentence description of a fake database:

**"The GlobalPet Registry is a fictional database containing records of over 50 million registered pets worldwide, including their names, species, medical histories, and owner information."**
